The contents of this notebook were created with assistance from Claude generative AI.

*requirements.txt* for this notebook:

/# Requirements for congestion_pricing_pipeline.ipynb
/# Python 3.12 (developed on 3.12.11)
/#
/# Install:  pip install -r requirements.txt
/#   (project standard is uv:  uv add requests pandas pyarrow ipykernel)
/#
/# Only third-party packages are listed; the notebook otherwise uses the standard
/# library (json, os, sys, time, datetime, pathlib).

requests==2.34.2     # Arctic Shift API collection (initial crawl + patches)
pandas==3.0.3        # DataFrames, JSONL reading, dedup, Parquet I/O
pyarrow==24.0.0      # Parquet engine, canonical schemas, batched combine
ipykernel==7.2.0     # run the notebook as a kernel in VSCode / Jupyter


# Building the NYC Reddit Corpus

*NOTE - This process was accomplished and documented with the assistance of GenAI (Claude Code).*

*UM MADS 696 - Milestone 2 - data-acquisition pipeline*

This notebook reproduces, end to end, how we built the two analysis-ready Parquet
files that back the NYC congestion-pricing stance/sentiment study:

| File | Rows | Size |
|------|-----:|-----:|
| `parquet/all_posts.parquet`    |    996,115 |   213.6 MB |
| `parquet/all_comments.parquet` | 18,408,220 | 1,992.9 MB |

Both files cover **2023-01-01 -> 2026-04-30** across ~58 NYC-area and
transit/urbanist subreddits, share one canonical schema, carry a `subreddit`
column on every row, and are deduplicated on the Reddit base-36 `id`. These **are not filtered** to only relevant posts/comments (about the CBDTP) yet.

### The pipeline so far at a glance

```
  Arctic Shift API --+
                     +--> per-subreddit ndjson / jsonl
  Download tool   ---+              |
                                    v
                   build per-subreddit Parquet
                                    |
                                    v
                   coverage / gap analysis
                                    |
                                    v
                   patch collection for identified gaps
                                    |
                                    v
                   merge patches into Parquet
                                    |
                                    v
                   combine -> all_posts.parquet / all_comments.parquet
```

### How to run this notebook:

The heavy stages (the two-day API crawl, the multi-GB Parquet build) are guarded by `RUN_*` flags that default to `False`, so you can read top-to-bottom without kicking off hours of work. Flip a flag to `True` to actually execute that stage, and run the cells in order.

## 1 - Data sources

Everything comes from **[Arctic Shift](https://github.com/ArthurHeitmann/arctic_shift)**,
a public archive of Reddit. We pulled from it two ways, because they have
different cost profiles:

**a) The Arctic Shift API** - `https://arctic-shift.photon-reddit.com`
Paginated search endpoints (`/api/posts/search`, `/api/comments/search`). Ideal
for small/medium subreddits and for ffetching specific date ranges.
We walk each query forward in time using `created_utc` as a cursor.

**b) The Arctic Shift download tool** - `https://arctic-shift.photon-reddit.com/download-tool`
Used for large subreddits (`AskNYC`, `astoria`, `bronx`, `Brooklyn`, `Connecticut`, `fuckcars`, `Hoboken`, `hudsonvalley`, `jerseycity`, `longisland`, `newjersey`, `newyork`, `newyorkcity`, `NYC`, `NYCbike`, `nycbus`, `NYCRail`, `parkslope`, `Queens`, `Ridgewood`, `transit`, `UpperWestSide`, `UrbanPlanning`, `westchester`), becuase thousands of paginated API calls would 
dominate the run, so we exported those subreddits as per-subreddit JSONL files
instead. The tool writes one file per `(subreddit, kind)`, e.g. 
`r_AskNYC_posts.jsonl`, `r_AskNYC_comments.jsonl`, into a local 
`searchtool download/` folder. **This is a manual/CLI step outside the
notebook** - you run the tool yourself and drop the JSONL into that folder.

The two sources overlap deliberately at the edges; deduplication on `id` makes
the overlap harmless.

**Time window.** Nothing newer than **2026-04-30** (the API/download tool upper bound is the exclusive `2026-05-01`) and nothing older than **2023-01-01**. We chose this date to align with available NYC Open Data.

**NSFW / media.** Posts flagged `over_18` are dropped. The archives are JSON only
(no image/video binaries), so instead of filtering media we *tag* each post with a
derived `_media_type` (`text`/`image`/`video`/`link`) for later slicing.

## 2 - Environment & the machine this ran on

The pipeline is **network- and I/O-bound**, not compute-bound: it spends its time
waiting on HTTP responses and streaming JSONL off disk. The original run was on:

- **OS:** Windows 11
- **Python:** 3.12, managed exclusively with **[uv](https://docs.astral.sh/uv/)**
  (`uv run python ...`, `uv add ...` - never bare `pip`/`python`)
- **RAM:** 96 GB (lets us hold a full subreddit's comments in a DataFrame during
  dedup)
- **GPU:** 2x NVIDIA RTX 4090 (48 GB VRAM total) - **idle for this notebook.** The
  GPUs matter for the *downstream* stance/sentiment modeling, not for collection
  or Parquet assembly.
- **Storage:** the raw archive (~40GB of JSONL) is cached on a local **HDD**.
  The project tree also lives under a Google Drive-synced path (`G:\...`), which is
  why every state/Parquet write below goes through an **atomic `os.replace`** and
  why the state saver retries on transient `OSError` (Drive briefly locks files
  mid-sync).

**Dependencies:** `requests`, `pandas`, `pyarrow`. One-time, from the project root:

```
uv add requests pandas pyarrow
```

**Rough runtimes** (vary with network + disk):

| Stage | Order of magnitude |
|-------|--------------------|
| Initial API crawl (all months x ~40 subs) | ~40 hours, resumable |
| Initial Download Tool use (manual) | ~1-15 hours, depending on subreddit size |
| Build per-subreddit Parquet from cached JSONL | ~10-20 min |
| Coverage analysis | ~1-2 min |
| Patch collection | minutes (only missing days) |
| Combine into mega-files | a few min |

In [1]:
import json
import os
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# -- Project layout -----------------------------------------------------------
# Point PROJECT at your own checkout if you are reproducing this elsewhere.
PROJECT   = Path("../data")

API_DIR   = PROJECT / "api_collection" / "congestion_pricing_data"  # initial crawl output
PATCH_DIR = PROJECT / "api_collection" / "patch_data"               # targeted patch output
TOOL_DIR  = PROJECT / "searchtool download"                         # download-tool JSONL
PARQUET   = PROJECT / "parquet"                                     # all Parquet output

for d in (API_DIR, PATCH_DIR, PARQUET / "posts", PARQUET / "comments"):
    d.mkdir(parents=True, exist_ok=True)

# -- Collection parameters ----------------------------------------------------
BASE         = "https://arctic-shift.photon-reddit.com"
EARLIEST     = "2023-01-01"   # inclusive lower bound
NEWEST       = "2026-05-01"   # EXCLUSIVE upper bound -> nothing newer than 2026-04-30
EXCLUDE_NSFW = True           # drop posts flagged over_18
DELAY        = 0.5            # seconds between API calls; raise if you see 429s

# -- Run guards: flip to True to actually execute a heavy stage ---------------
RUN_INITIAL_COLLECTION = False   # the overnight API crawl
RUN_BUILD_PARQUET      = False   # build per-subreddit Parquet from JSONL
RUN_PATCH_COLLECTION   = False   # fetch the gap days
RUN_MERGE              = False   # fold patches + tool gaps into Parquet
RUN_COMBINE            = False   # build the two mega-files

print("Project root:", PROJECT)
print("Exists:", PROJECT.exists())

Project root: Project Data
Exists: True


## 3 - The subreddit universe

Two lists. `API_SUBREDDITS` are collected through the API; the heavy hitters in
`DOWNLOAD_TOOL_SUBREDDITS` are exported with the download tool instead (the API
list intentionally omits them). A few subs appear in both because they were
partly sourced each way - dedup on `id` reconciles them. Names are
case-insensitive to the API; empty or nonexistent subs simply yield nothing, so
over-including is cheap. The set spans the congestion zone, the five boroughs,
active neighborhood subs, NYC transit/cycling subs, and the NY/CT/NJ commuter
shed.

In [2]:
# Large subreddits - exported via the download tool (NOT the API), because a bulk
# per-subreddit dump is far quicker than thousands of paginated calls.
DOWNLOAD_TOOL_SUBREDDITS = [
    "nyc", "newjersey", "AskNYC", "longisland", "Brooklyn", "Queens",
    "Connecticut", "transit", "fuckcars", "UrbanPlanning",
    "newyork", "newyorkcity", "bronx", "Hoboken", "jerseycity", "parkslope",
    "astoria", "Ridgewood", "NYCbike", "nycrail", "nycbus", "UpperWestSide",
    "Westchester", "HudsonValley", 
]

# Everything else - collected through the API, month by month.
API_SUBREDDITS = [
    # NYC citywide / boroughs
    "manhattan", "StatenIsland",
    # Manhattan neighborhoods
    "uppereastside", "EastVillage", "washingtonheights", "Inwood", "Harlem",
    "RooseveltIsland",
    # Brooklyn neighborhoods
    "Williamsburg", "Greenpoint", "Bushwick", "bedstuy", "crownheights",
    "sunsetpark", "BayRidge",
    # Queens neighborhoods
    "longislandcity", "jacksonheights", "foresthills", "Flushing",
    # NYC transit / cycling
    "MTA", "MicromobilityNYC",
    # NY state + Lower Hudson Valley / LI commuter shed
    "rockland", "Yonkers", "WhitePlains", "NewRochelle",
    # Connecticut commuter towns (Metro-North)
    "Stamford", "Greenwich",
    # New Jersey commuter shed
    "bergencounty", "HudsonCountyNJ", "njtransit", "Newark", "FortLee",
    "Montclair", "Hackensack", "NewBrunswick",
    
]

print(f"{len(API_SUBREDDITS)} API subreddits, "
      f"{len(DOWNLOAD_TOOL_SUBREDDITS)} download-tool subreddits")

35 API subreddits, 24 download-tool subreddits


## 4 - A shared Arctic Shift API client

The initial crawl and the later patch collection use the *same* HTTP machinery,
so we define it once:

- **`api_get`** - GET with retry + back-off. Handles `429` (rate limit), `5xx`,
  transient network errors, and honors the `X-RateLimit-Remaining` header.
- **`paginate`** - walks a search endpoint over `[after, before)` using
  `created_utc` as a forward cursor; stops when a page returns no new IDs.
- **`classify_media`** - derives `_media_type` for posts.

Pagination omits `limit` so the API serves up to ~1000 records/page for
throughput. `load_state` / `write_state` make every long crawl resumable by
recording finished work units, with atomic, retrying writes for Drive safety.

In [3]:
session = requests.Session()
session.headers.update({"User-Agent": "congestion-pricing-research/1.0 (academic)"})


def api_get(path, params):
    """GET with retry + rate-limit back-off. Returns the list of records."""
    url = BASE + path
    for attempt in range(6):
        try:
            r = session.get(url, params=params, timeout=180)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        if r.status_code != 200:
            print(f"    ! {r.status_code} on {path} {params}: {r.text[:160]}")
            return []
        rem = r.headers.get("X-RateLimit-Remaining")
        if rem is not None:
            try:
                if int(rem) <= 1:
                    time.sleep(5)
            except ValueError:
                pass
        time.sleep(DELAY)
        body = r.json()
        if isinstance(body, dict):
            return body.get("data", [])
        return body if isinstance(body, list) else []
    print(f"    ! giving up on {path} {params}")
    return []


def paginate(path, base_params, after, before):
    """Page through [after, before) using created_utc as a forward cursor."""
    p = dict(base_params)
    p["before"] = before
    p["sort"]   = "asc"
    cursor = after
    seen   = set()
    while True:
        p["after"] = cursor
        batch = api_get(path, p)
        if not batch:
            return
        new = [x for x in batch if x.get("id") not in seen]
        if not new:
            return
        for x in new:
            seen.add(x["id"])
            yield x
        last_ts = batch[-1].get("created_utc")
        if last_ts is None:
            return
        cursor = last_ts


def classify_media(post):
    if post.get("is_self"):
        return "text"
    hint = (post.get("post_hint") or "").lower()
    if "image" in hint:
        return "image"
    if "video" in hint or post.get("is_video"):
        return "video"
    url = (post.get("url") or "").lower()
    if any(url.endswith(ext) for ext in (".jpg", ".jpeg", ".png", ".gif", ".gifv")):
        return "image"
    if any(d in url for d in ("i.redd.it", "imgur.com")):
        return "image"
    if any(d in url for d in ("v.redd.it", "youtube.com", "youtu.be")):
        return "video"
    if post.get("url"):
        return "link"
    return "text"


# The two endpoints we page, and whether each is a "post" (needs media tagging).
KINDS = (
    ("posts",    "/api/posts/search",    True),
    ("comments", "/api/comments/search", False),
)


def load_state(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as fh:
            return set(json.load(fh))
    return set()


def write_state(path, completed):
    """Atomic state write; retry to survive Drive-sync file locks."""
    tmp = str(path) + ".tmp"
    for attempt in range(5):
        try:
            with open(tmp, "w", encoding="utf-8") as fh:
                json.dump(sorted(completed), fh)
            os.replace(tmp, path)
            return
        except OSError as e:
            if attempt < 4:
                time.sleep(1 + attempt)
            else:
                print(f"    ! state save failed after 5 attempts: {e} -- continuing")


print("API client ready.")

API client ready.


## 5 - Step 1: Initial API crawl (month by month, newest -> oldest)

For every `API_SUBREDDITS` entry we download **all** posts and **all** comments in
the window, one calendar month at a time, walking backward from the newest month.
Output is one ndjson file per `(kind, subreddit, month)`:

```
api_collection/congestion_pricing_data/
    posts/<subreddit>/<YYYY-MM>.ndjson
    comments/<subreddit>/<YYYY-MM>.ndjson
    _completed.json          # finished (kind, sub, month) units -> resumable
```

The crawl is **resumable**: each finished `(kind, sub, month)` is recorded, so a
re-run (or recovery after Ctrl-C / a dropped connection) skips what's done. This
is the multi-hour stage - set `RUN_INITIAL_COLLECTION = True` and let it run
over ~2 days.

In [4]:
INIT_STATE = API_DIR / "_completed.json"


def month_windows(earliest, newest):
    """Consecutive [start, end) month boundaries covering [earliest, newest), newest-first."""
    y0, m0 = int(earliest[:4]), int(earliest[5:7])
    yN, mN = int(newest[:4]),  int(newest[5:7])
    bounds, (y, m) = [], (y0, m0)
    while (y, m) <= (yN, mN):
        bounds.append(f"{y:04d}-{m:02d}-01")
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return list(reversed(list(zip(bounds, bounds[1:]))))


def run_initial_collection(subreddits):
    completed = load_state(INIT_STATE)
    if completed:
        print(f"Resuming: {len(completed)} (kind, sub, month) units already done.")
    windows = month_windows(EARLIEST, NEWEST)
    print(f"Collecting {len(subreddits)} subreddits x {len(windows)} months, "
          f"newest -> oldest, capped at {NEWEST} (exclusive).\n")

    for start, end in windows:
        label = start[:7]                       # "YYYY-MM"
        print(f"===== {label} =====")
        month_total = 0
        for sub in subreddits:
            for kind, path, is_post in KINDS:
                key = f"{kind}|{sub.lower()}|{label}"
                if key in completed:
                    continue
                outpath = API_DIR / kind / sub / f"{label}.ndjson"
                outpath.parent.mkdir(parents=True, exist_ok=True)
                n = 0
                with open(outpath, "w", encoding="utf-8") as fh:
                    for rec in paginate(path, {"subreddit": sub}, start, end):
                        if is_post:
                            if EXCLUDE_NSFW and rec.get("over_18"):
                                continue
                            rec["_media_type"] = classify_media(rec)
                        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        n += 1
                completed.add(key)
                write_state(INIT_STATE, completed)
                month_total += n
                if n:
                    print(f"    r/{sub} {kind}: {n}")
        print(f"  -- {label} total: {month_total}\n")
    print("Initial collection done.")


if RUN_INITIAL_COLLECTION:
    run_initial_collection(API_SUBREDDITS)
else:
    print("RUN_INITIAL_COLLECTION is False -- skipping the overnight crawl.")
    print(f"Would collect {len(API_SUBREDDITS)} subs across "
          f"{len(month_windows(EARLIEST, NEWEST))} months.")

RUN_INITIAL_COLLECTION is False -- skipping the overnight crawl.
Would collect 35 subs across 40 months.


## 6 - Step 2: Large subreddits via the download tool

The subreddits in `DOWNLOAD_TOOL_SUBREDDITS` are pulled with the Arctic Shift
**download tool** rather than the API, then dropped into `searchtool download/` as
per-subreddit JSONL:

```
searchtool download/
    r_AskNYC_posts.jsonl      r_AskNYC_comments.jsonl
    r_NYC_posts.jsonl         r_NYC_comments.jsonl
    r_fuckcars_posts.jsonl    r_fuckcars_comments.jsonl
    ...
```

A couple of practical notes (kept brief on purpose):

- **Re-downloads happen.** Some big comment files were exported in pieces or
  re-pulled to close a gap, so you'll see `..._comments2.jsonl` /
  `..._comments3.jsonl` siblings. They are just additional inputs - dedup on `id`
  collapses any overlap, so listing several files for one subreddit is safe.
- A couple of filenames carry quirks (a doubled `r_r_` prefix, lowercase
  spellings); we simply point at the real path.

The mapping below tells the build step which tool files belong to which
subreddit. Files that are not on disk are skipped with a warning. (The
`...-gap.jsonl` and doubled-prefix files are intentionally **not** here - they are
folded in later, in the merge step, because that is when they were pulled.)

In [5]:
T = TOOL_DIR

# {subreddit: {"posts": [Path, ...], "comments": [Path, ...]}}
# Multiple files per kind = original export + re-downloads (order = dedup priority).
TOOL_SOURCES = {
    "AskNYC":       {"posts":    [T / "r_AskNYC_posts.jsonl"],
                     "comments": [T / "r_AskNYC_comments.jsonl",
                                  T / "r_AskNYC_comments2.jsonl",
                                  T / "r_AskNYC_comments3.jsonl"]},
    "Brooklyn":     {"posts":    [T / "r_Brooklyn_posts.jsonl"],
                     "comments": [T / "r_Brooklyn_comments.jsonl"]},
    "Connecticut":  {"posts":    [T / "r_Connecticut_posts.jsonl"],
                     "comments": [T / "r_Connecticut_comments.jsonl",
                                  T / "r_Connecticut_comments2.jsonl",
                                  T / "r_Connecticut_comments3.jsonl"]},
    "fuckcars":     {"posts":    [T / "r_fuckcars_posts.jsonl"],
                     "comments": [T / "r_fuckcars_comments.jsonl",
                                  T / "r_fuckcars_comments2.jsonl",
                                  T / "r_fuckcars_comments3.jsonl"]},
    "longisland":   {"posts":    [T / "r_longisland_posts.jsonl"],
                     "comments": [T / "r_longisland_comments.jsonl"]},
    "newjersey":    {"posts":    [T / "r_newjersey_posts.jsonl"],
                     "comments": [T / "r_newjersey_comments.jsonl",
                                  T / "r_newjersey_comments2.jsonl"]},
    "nyc":          {"posts":    [T / "r_NYC_posts.jsonl"],
                     "comments": [T / "r_NYC_comments.jsonl",
                                  T / "r_NYC_comments2.jsonl"]},
    "nycbus":       {"posts":    [T / "r_nycbus_posts.jsonl"],
                     "comments": [T / "r_nycbus_comments.jsonl"]},
    "Queens":       {"posts":    [T / "r_Queens_posts.jsonl"],
                     "comments": [T / "r_Queens_comments.jsonl"]},
    "transit":      {"posts":    [T / "r_transit_posts.jsonl"],
                     "comments": [T / "r_transit_comments.jsonl"]},
    "UrbanPlanning":{"posts":    [T / "r_UrbanPlanning_posts.jsonl"],
                     "comments": [T / "r_UrbanPlanning_comments.jsonl"]},
    "astoria":      {"posts":    [T / "r_astoria_posts.jsonl",
                                  T / "r_astoria_posts2.jsonl"],
                     "comments": [T / "r_astoria_comments.jsonl",
                                  T / "r_astoria_comments2.jsonl"]},
    "bronx":        {"posts":    [T / "r_bronx_posts.jsonl"],
                     "comments": [T / "r_bronx_comments.jsonl"]},
    "Hoboken":      {"posts":    [T / "r_Hoboken_posts.jsonl"],
                     "comments": [T / "r_Hoboken_comments.jsonl"]},
    "HudsonValley": {"posts":    [T / "r_hudsonvalley_posts.jsonl",
                                  T / "r_hudsonvalley_posts2.jsonl"],
                     "comments": [T / "r_hudsonvalley_comments.jsonl",
                                  T / "r_hudsonvalley_comments2.jsonl"]},
    "jerseycity":   {"posts":    [T / "r_jerseycity_posts.jsonl",
                                  T / "r_jerseycity_posts2.jsonl"],
                     "comments": [T / "r_jerseycity_comments.jsonl",
                                  T / "r_jerseycity_comments2.jsonl"]},
    "newyork":      {"posts":    [T / "r_newyork_posts.jsonl"],
                     "comments": [T / "r_newyork_comments.jsonl"]},
    "newyorkcity":  {"posts":    [T / "r_newyorkcity_posts.jsonl"],
                     "comments": [T / "r_newyorkcity_comments.jsonl"]},
    "NYCbike":      {"posts":    [T / "r_NYCbike_posts.jsonl"],
                     "comments": [T / "r_NYCbike_comments.jsonl"]},
    "nycrail":      {"posts":    [T / "r_NYCRail_posts.jsonl"],
                     "comments": [T / "r_NYCRail_comments.jsonl"]},
    "parkslope":    {"posts":    [T / "r_parkslope_posts.jsonl"],
                     "comments": [T / "r_parkslope_comments.jsonl"]},
    "Ridgewood":    {"posts":    [T / "r_Ridgewood_posts.jsonl"],
                     "comments": [T / "r_Ridgewood_comments.jsonl"]},
    "UpperWestSide":{"posts":    [T / "r_UpperWestSide_posts.jsonl"],
                     "comments": [T / "r_UpperWestSide_comments.jsonl"]},
    "Westchester":  {"posts":    [T / "r_westchester_posts.jsonl",
                                  T / "r_westchester_posts2.jsonl"],
                     "comments": [T / "r_westchester_comments.jsonl",
                                  T / "r_westchester_comments2.jsonl"]},
}

print(f"{len(TOOL_SOURCES)} subreddits mapped to download-tool files.")

24 subreddits mapped to download-tool files.


## 7 - Step 3: Build clean per-subreddit Parquet

Now we fold both sources into one tidy Parquet per `(subreddit, kind)`:

```
parquet/posts/<subreddit>.parquet
parquet/comments/<subreddit>.parquet
```

For each subreddit we read its API monthly ndjson **and** its download-tool JSONL,
keep only the columns in our feature inventory
(`Design/feature_inventory_posts.csv`, `..._comments.csv`), concatenate, and
**deduplicate on `id`** (first file listed wins - API monthly files are placed
before tool files). One subreddit, `HellsKitchen`, is dropped: it is swamped by
the TV show, not the neighborhood.

**Type hygiene (`prepare_for_parquet`).** Reddit JSON has a few columns pyarrow
cannot serialize directly:

- `poll_data` holds dicts/lists (and occasionally stray bools) -> JSON-stringify.
- `edited` is `False` *or* a Unix timestamp -> map `False`->null, store as nullable
  `Int64`.
- a column that is mostly strings but picks up raw bools/ints from a second source
  -> JSON-stringify the whole column.

We always write to a `.parquet.tmp` then `os.replace`, so a crash never leaves a
half-written file.

In [6]:
POST_COLS = [
    "id", "name", "created_utc", "retrieved_on",
    "title", "selftext", "previous_selftext", "url", "domain",
    "permalink", "author", "author_fullname", "author_created_utc",
    "subreddit", "subreddit_id", "subreddit_subscribers",
    "score", "ups", "upvote_ratio", "num_comments",
    "is_self", "_media_type", "post_hint", "over_18",
    "locked", "stickied", "distinguished", "edited",
    "removed_by_category", "link_flair_text", "poll_data",
]

COMMENT_COLS = [
    "id", "name", "created_utc", "retrieved_on",
    "body", "permalink", "author", "author_fullname", "author_created_utc",
    "subreddit", "subreddit_id",
    "link_id", "parent_id",
    "score", "ups", "controversiality",
    "edited", "distinguished", "is_submitter",
    "stickied", "locked", "collapsed",
    "removed_by_category",
]

COLS = {"posts": POST_COLS, "comments": COMMENT_COLS}
EXCLUDE_SUBS = {"HellsKitchen"}   # dominated by the TV show, not the neighborhood
print(f"{len(POST_COLS)} post columns, {len(COMMENT_COLS)} comment columns")

31 post columns, 23 comment columns


In [7]:
def read_jsonl(path, cols):
    """Read one .jsonl/.ndjson in 50k-row chunks, keeping only wanted columns."""
    chunks = []
    try:
        for chunk in pd.read_json(path, lines=True, chunksize=50_000):
            keep = [c for c in cols if c in chunk.columns]
            chunks.append(chunk[keep])
    except Exception as exc:
        print(f"    ERROR reading {path.name}: {exc}", file=sys.stderr)
        return pd.DataFrame()
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()


def write_parquet_safe(df, out_path):
    """Write to a .tmp then atomically replace, so crashes never corrupt the original."""
    tmp = out_path.with_suffix(".parquet.tmp")
    df.to_parquet(tmp, engine="pyarrow", compression="zstd", index=False)
    os.replace(tmp, out_path)   # atomic on Windows; overwrites existing file


def prepare_for_parquet(df):
    """Fix mixed-Python-type object columns that pyarrow cannot serialize directly."""
    for col in df.select_dtypes(include=["object", "str"]).columns:
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        has_complex = non_null.apply(lambda x: isinstance(x, (dict, list))).any()
        has_string  = non_null.apply(lambda x: isinstance(x, str)).any()
        has_bool    = non_null.apply(lambda x: isinstance(x, bool)).any()
        has_number  = non_null.apply(lambda x: isinstance(x, (int, float)) and not isinstance(x, bool)).any()

        def _to_json(x):
            if x is None:
                return None
            try:
                if pd.isna(x):
                    return None
            except (TypeError, ValueError):
                pass
            if isinstance(x, str):
                return x
            return json.dumps(x, ensure_ascii=False)

        if has_complex or (has_string and (has_bool or has_number)):
            # dicts/lists -> JSON strings; also covers a string column that picked
            # up raw bools/ints from a second source (e.g. serialized poll_data).
            df[col] = df[col].apply(_to_json)
        elif has_bool and has_number:
            # e.g. edited: False | 1672608506 -> pd.NA | 1672608506
            df[col] = df[col].apply(lambda x: pd.NA if x is False else x)
            df[col] = pd.array(df[col], dtype="Int64")
    return df


def collect_sources(kind):
    """{subreddit: [Path, ...]} = API monthly ndjson (auto-discovered) + tool files."""
    sources = {}
    # Step 1 - API collection (one folder per subreddit, monthly ndjson)
    api_kind_dir = API_DIR / kind
    if api_kind_dir.exists():
        for sub_dir in sorted(api_kind_dir.iterdir()):
            if sub_dir.is_dir():
                files = sorted(sub_dir.glob("*.ndjson"))
                if files:
                    sources[sub_dir.name] = list(files)
    # Step 2 - download-tool files (API files stay first so they win on dedup)
    for sub_name, file_map in TOOL_SOURCES.items():
        found = [p for p in file_map.get(kind, []) if p.exists()]
        for p in (p for p in file_map.get(kind, []) if not p.exists()):
            print(f"  [!] Skipping (not found): {p.name}")
        if not found and sub_name not in sources:
            continue
        sources.setdefault(sub_name, [])
        sources[sub_name] = sources[sub_name] + found
    # Step 3 - exclusions
    for sub in EXCLUDE_SUBS:
        sources.pop(sub, None)
    return sources


def process_sub(sub_name, files, cols, kind):
    """Read, combine, dedup, and write Parquet for one (subreddit, kind) pair."""
    dfs = []
    for f in files:
        df = read_jsonl(f, cols)
        if not df.empty:
            dfs.append(df)
    if not dfs:
        print(f"  [{sub_name}] no data -- skipping.")
        return
    combined = pd.concat(dfs, ignore_index=True)
    before = len(combined)
    combined.drop_duplicates(subset=["id"], keep="first", inplace=True)
    after = len(combined)
    combined = prepare_for_parquet(combined)
    out_path = PARQUET / kind / f"{sub_name}.parquet"
    write_parquet_safe(combined, out_path)
    size_mb = out_path.stat().st_size / 1_048_576
    print(f"  [{sub_name}] {after:,} records ({before - after:,} dupes dropped), {size_mb:.1f} MB")


def build_all_parquet():
    for kind, cols in (("posts", POST_COLS), ("comments", COMMENT_COLS)):
        print(f"\n===== {kind.upper()} =====")
        sources = collect_sources(kind)
        print(f"{len(sources)} subreddits with {kind} data.\n")
        for sub_name, files in sorted(sources.items()):
            process_sub(sub_name, files, cols, kind)
    print("\nBuild done.")


if RUN_BUILD_PARQUET:
    build_all_parquet()
else:
    print("RUN_BUILD_PARQUET is False -- skipping the per-subreddit build.")

RUN_BUILD_PARQUET is False -- skipping the per-subreddit build.


## 8 - A look at what we built

Before going further, sanity-check a per-subreddit file: size, schema, a few rows,
and date span. (Reads are cheap - these cells run regardless of the `RUN_*` flags,
as long as the Parquet exists.)

In [8]:
def peek(sub, kind="comments", n=3):
    path = PARQUET / kind / f"{sub}.parquet"
    if not path.exists():
        print(f"(no file yet: {path.name})")
        return None
    pf = pq.ParquetFile(path)
    print(f"{kind}/{sub}.parquet  -  {pf.metadata.num_rows:,} rows, "
          f"{path.stat().st_size / 1_048_576:.1f} MB")
    print("columns:", [f.name for f in pf.schema_arrow])
    df = pd.read_parquet(path)
    ts = pd.to_datetime(pd.to_numeric(df["created_utc"], errors="coerce"), unit="s", utc=True)
    print("date span:", ts.min().date(), "->", ts.max().date())
    return df.head(n)


peek("AskNYC", "comments")

comments/AskNYC.parquet  -  1,895,712 rows, 217.7 MB
columns: ['id', 'name', 'created_utc', 'retrieved_on', 'body', 'permalink', 'author', 'author_fullname', 'author_created_utc', 'subreddit', 'subreddit_id', 'link_id', 'parent_id', 'score', 'ups', 'controversiality', 'edited', 'distinguished', 'is_submitter', 'stickied', 'locked', 'collapsed']
date span: 2023-01-01 -> 2026-04-30


,id,name,created_utc,retrieved_on,body,permalink,author,author_fullname,author_created_utc,subreddit,...,parent_id,score,ups,controversiality,edited,distinguished,is_submitter,stickied,locked,collapsed
0,j2fxu2x,t1_j2fxu2x,1672531275,1.673022e+09,I live just a short drive from the Babyland Ge...,/r/AskNYC/comments/zykwxa/where_in_the_city_ca...,ReasonableDust2164,t2_b8h0tx3g,1.617127e+09,AskNYC,...,t1_j29abqw,2,2,0,1672531540,NaN,False,False,False,False
1,j2fxxr2,t1_j2fxxr2,1672531324,1.673022e+09,"You should really, really think about whether ...",/r/AskNYC/comments/1005i8l/moving_out_of_nyc_w...,WelcomeToBrooklandia,t2_3ksg2agw,1.582823e+09,AskNYC,...,t3_1005i8l,15,15,0,0,NaN,False,False,False,False
2,j2fy2g8,t1_j2fy2g8,1672531384,1.673022e+09,THIS!!! I have not had to have this convo mu...,/r/AskNYC/comments/zzdeit/new_upstairs_neighbo...,ApartNefariousness95,t2_ab7jdgyq,1.613115e+09,AskNYC,...,t1_j2esgt3,2,2,0,0,NaN,False,False,False,False


## 9 - Step 4: Coverage / gap analysis

Are there date holes? We probe with **comments** (denser than posts, so they
reveal holes better). For each comments Parquet we compute, over
2023-01-01 -> 2026-04-30: total rows, days missing, the longest run of consecutive
missing days, and the median comments/day on active days. A gap is flagged
**suspicious** when the sub is active enough that we would expect daily data
(median >= 5/day) yet still shows a run of >= 14 missing days - that pattern smells
like a *data* hole rather than a quiet subreddit.

The next cell computes this report directly from the Parquet files.

In [9]:
COV_START, COV_END = date(2023, 1, 1), date(2026, 4, 30)


def _longest_gap(missing):
    """(length, start, end) of the longest run of consecutive missing dates."""
    if not missing:
        return 0, None, None
    best_len, best_s, best_e = 1, missing[0], missing[0]
    cur_len, cur_s = 1, missing[0]
    for i in range(1, len(missing)):
        if (missing[i] - missing[i - 1]).days == 1:
            cur_len += 1
        else:
            if cur_len > best_len:
                best_len, best_s, best_e = cur_len, cur_s, missing[i - 1]
            cur_len, cur_s = 1, missing[i]
    if cur_len > best_len:
        best_len, best_s, best_e = cur_len, cur_s, missing[-1]
    return best_len, best_s, best_e


def coverage_report(comments_dir):
    all_days, d = set(), COV_START
    while d <= COV_END:
        all_days.add(d)
        d += timedelta(days=1)

    rows = []
    for pf in sorted(Path(comments_dir).glob("*.parquet")):
        ts = pq.read_table(pf, columns=["created_utc"]).column("created_utc").to_pandas()
        ts = pd.to_numeric(ts, errors="coerce").dropna().astype("int64")
        dates = pd.to_datetime(ts, unit="s", utc=True).dt.date
        in_window = dates[(dates >= COV_START) & (dates <= COV_END)]
        daily = in_window.value_counts()
        missing = sorted(all_days - set(daily.index))
        glen, gs, ge = _longest_gap(missing)
        med = round(float(daily.median()), 1) if len(daily) else 0.0
        rows.append(dict(
            sub=pf.stem, total=len(ts), days_missing=len(missing),
            max_gap=glen, gap_start=gs, gap_end=ge, median_per_day=med,
            suspicious=(glen >= 14 and med >= 5),
        ))
    return pd.DataFrame(rows).sort_values("total", ascending=False).reset_index(drop=True)


pd.set_option("display.max_rows", 200)
cov = coverage_report(PARQUET / "comments")
print(f"{len(cov)} subreddits checked; "
      f"{int(cov['suspicious'].sum())} flagged suspicious.\n")
cov

58 subreddits checked; 15 flagged suspicious.



,sub,total,days_missing,max_gap,gap_start,gap_end,median_per_day,suspicious
0,nyc,2177243,0,0,None,None,1681.5,False
1,fuckcars,1962585,1,1,2023-06-14,2023-06-14,1550.0,False
2,newjersey,1897458,7,7,2023-06-12,2023-06-18,1493.0,False
3,AskNYC,1895712,0,0,None,None,1499.0,False
4,Connecticut,1848310,2,2,2023-06-13,2023-06-14,1456.0,False
5,longisland,1188778,7,7,2023-06-13,2023-06-19,945.0,False
6,jerseycity,764790,0,0,None,None,602.0,False
7,nycrail,595497,0,0,None,None,455.5,False
8,Brooklyn,571352,1,1,2023-06-13,2023-06-13,416.0,False
9,transit,537968,0,0,None,None,422.0,False


In [10]:
# Just the flagged gaps, longest first - these drive the patch list.
cov[cov["suspicious"]].sort_values("max_gap", ascending=False)

,sub,total,days_missing,max_gap,gap_start,gap_end,median_per_day,suspicious
47,washingtonheights,2010,1008,313,2023-11-25,2024-10-02,5.0,True
49,sunsetpark,949,1109,264,2023-08-14,2024-05-03,6.0,True
42,EastVillage,14890,586,131,2023-12-28,2024-05-06,13.0,True
27,UpperWestSide,105224,223,57,2023-01-10,2023-03-07,87.0,True
16,Bushwick,233093,56,28,2025-02-01,2025-02-28,189.0,True
37,crownheights,37262,318,28,2023-01-29,2023-02-25,24.0,True
33,bedstuy,63210,264,27,2025-02-02,2025-02-28,51.0,True
18,Williamsburg,220321,130,26,2026-04-05,2026-04-30,183.0,True
30,bergencounty,96514,80,26,2025-04-05,2025-04-30,58.0,True
23,Newark,126552,82,24,2025-07-08,2025-07-31,96.0,True


### What the report tells us

Two kinds of holes show up:

1. **Recency edge.** A handful of download-tool exports stopped a few weeks before
   2026-04-30 (the April archive might have landed only after those pulls). Those tail days
   are real data we can still fetch.
2. **Interior gaps.** The flagged runs inside the window.

Not every gap is fillable - some are genuine absences that the API also returns 0
for, and we should *not* keep chasing:

- **The June 2023 Reddit blackout** (Jun 12-18, 2023): thousands of subs went
  private protesting Reddit's API pricing. `Connecticut` (Jun 13-14),
  `newjersey` (Jun 12-18), and others are simply empty then.
- **Dormant subs.** `EastVillage` is empty Dec 2023 -> May 2024 (~131 days): the
  sub was inactive, confirmed by the API also returning 0.
- **Naturally sparse subs** (washingtonheights, sunsetpark, FortLee, ...): small
  gaps are expected, not errors.

Everything else - recency tails plus real interior holes in active subs - goes
into a targeted patch list.

## 10 - Step 5: Targeted patch collection (exact missing days)

We re-fetch **only the missing date ranges** - never whole months - reusing the
same `paginate`/`api_get` client. Output mirrors the crawl but in its own folder
so nothing collides:

```
api_collection/patch_data/
    posts/<sub>/<start>_to_<end>.ndjson
    comments/<sub>/<start>_to_<end>.ndjson
    _patch_completed.json
```

The `PATCHES` list below is the union of what our coverage checks surfaced. Each
tuple is `(subreddit, gap_start, gap_end_inclusive)` and is fetched for **both**
posts and comments.

> **Sidenote - why this was really two rounds.** In practice we ran
> collect -> merge -> re-check *twice*. Filling one hole (and folding in the
> download-tool gap files in the next step) shifted a few subreddit edges and
> exposed adjacent gaps, and a few very dense comment ranges timed out
> (HTTP 422 "slow down") and were retried. The consolidated list here reaches the
> same coverage in a single pass; the two original lists live in the repo as
> `collect_patches.py` / `collect_patches2.py`. A few entries here intentionally
> target ranges the API returns 0 for (blackout / dormant) - harmless, and they
> document that we checked.

In [11]:
PATCH_STATE = PATCH_DIR / "_patch_completed.json"

# (subreddit, gap_start, gap_end_inclusive) -- exact days only.
PATCHES = [
    # -- Recency tails: download-tool exports that stopped before 2026-04-30 --
    ("astoria",          "2026-03-21", "2026-04-30"),
    ("Greenpoint",       "2026-04-02", "2026-04-30"),
    ("HudsonValley",     "2026-04-19", "2026-04-30"),
    ("jerseycity",       "2026-04-08", "2026-04-30"),
    ("longislandcity",   "2026-04-03", "2026-04-30"),
    ("Westchester",      "2026-04-09", "2026-04-30"),
    # -- Interior gaps from the first coverage pass --
    ("bedstuy",          "2024-07-04", "2024-07-31"),
    ("bergencounty",     "2025-07-01", "2025-07-31"),
    ("Brooklyn",         "2024-02-14", "2024-03-25"),
    ("Bushwick",         "2024-06-02", "2024-06-30"),
    ("Connecticut",      "2023-06-13", "2023-06-14"),   # blackout -> returns 0
    ("crownheights",     "2023-01-29", "2023-02-25"),
    ("EastVillage",      "2023-12-28", "2024-05-06"),   # dormant -> returns 0
    ("Flushing",         "2023-06-01", "2023-06-30"),
    ("fuckcars",         "2023-02-17", "2023-02-23"),
    ("jacksonheights",   "2023-05-10", "2023-05-23"),
    ("longisland",       "2023-09-16", "2023-09-27"),
    ("manhattan",        "2025-06-11", "2025-06-30"),
    ("MicromobilityNYC", "2024-06-06", "2024-06-30"),
    ("Montclair",        "2025-07-01", "2025-07-31"),
    ("newyork",          "2025-02-23", "2025-03-25"),
    ("newjersey",        "2023-06-12", "2023-06-18"),   # blackout -> returns 0
    ("Newark",           "2025-04-04", "2025-04-30"),
    ("njtransit",        "2025-07-01", "2025-07-31"),
    ("nycbus",           "2023-03-07", "2023-03-28"),
    ("nycrail",          "2023-04-14", "2023-05-10"),
    ("StatenIsland",     "2025-06-01", "2025-06-30"),
    ("uppereastside",    "2025-10-02", "2025-10-31"),
    ("UpperWestSide",    "2023-01-10", "2023-03-07"),
    ("UrbanPlanning",    "2023-12-17", "2024-01-20"),
    ("Williamsburg",     "2025-10-01", "2025-10-31"),
    # -- Adjacent gaps revealed on the second coverage pass --
    ("Bushwick",         "2026-04-01", "2026-04-30"),
    ("uppereastside",    "2024-06-02", "2024-06-30"),
    ("uppereastside",    "2023-06-26", "2023-07-06"),
    ("bergencounty",     "2025-01-01", "2025-01-31"),
    ("Greenpoint",       "2024-06-04", "2024-06-30"),
    ("Greenpoint",       "2026-02-01", "2026-02-28"),
    ("bedstuy",          "2024-06-01", "2024-06-30"),
    ("njtransit",        "2024-08-04", "2024-08-31"),
    ("StatenIsland",     "2025-03-04", "2025-03-31"),
    ("Williamsburg",     "2025-05-06", "2025-05-31"),
    ("longislandcity",   "2025-02-03", "2025-02-28"),
    ("longislandcity",   "2025-06-04", "2025-06-30"),
    ("Newark",           "2023-06-07", "2023-06-30"),
    ("Montclair",        "2025-11-06", "2025-11-30"),
    ("Montclair",        "2023-05-31", "2023-06-30"),
    ("MicromobilityNYC", "2024-01-18", "2024-01-31"),
    ("MicromobilityNYC", "2024-05-11", "2024-05-31"),
    ("Westchester",      "2026-04-14", "2026-04-30"),
    ("manhattan",        "2025-10-17", "2025-10-31"),
    ("crownheights",     "2024-06-01", "2024-06-30"),
    ("nycbus",           "2023-03-14", "2023-03-29"),
    ("jacksonheights",   "2023-05-14", "2023-05-27"),
]


def day_after(yyyy_mm_dd):
    return (date.fromisoformat(yyyy_mm_dd) + timedelta(days=1)).isoformat()


def run_patches(patches):
    completed = load_state(PATCH_STATE)
    total_days = sum((date.fromisoformat(e) - date.fromisoformat(s)).days + 1
                     for _, s, e in patches)
    print(f"Patch plan: {len(patches)} gaps, "
          f"{len({s for s, _, _ in patches})} subreddits, {total_days} calendar days.\n")
    for sub, gs, ge in patches:
        before = day_after(ge)               # exclusive upper bound for the API
        label = f"{gs}_to_{ge}"
        print(f"  [{sub}] {gs} -> {ge}")
        for kind, path, is_post in KINDS:
            key = f"{kind}|{sub.lower()}|{label}"
            if key in completed:
                print(f"    skip {kind} (done)")
                continue
            outpath = PATCH_DIR / kind / sub / f"{label}.ndjson"
            outpath.parent.mkdir(parents=True, exist_ok=True)
            n = 0
            with open(outpath, "w", encoding="utf-8") as fh:
                for rec in paginate(path, {"subreddit": sub}, gs, before):
                    if is_post:
                        if EXCLUDE_NSFW and rec.get("over_18"):
                            continue
                        rec["_media_type"] = classify_media(rec)
                    fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    n += 1
            completed.add(key)
            write_state(PATCH_STATE, completed)
            print(f"    {kind}: {n:,}")
    print("Patches done.")


if RUN_PATCH_COLLECTION:
    run_patches(PATCHES)
else:
    print(f"RUN_PATCH_COLLECTION is False -- skipping. {len(PATCHES)} gaps queued.")

RUN_PATCH_COLLECTION is False -- skipping. 53 gaps queued.


## 11 - Step 6: Merge patches and tool-gap files into the Parquet

Two kinds of "extra" JSONL now get folded into the per-subreddit Parquet:

1. **API patches** in `patch_data/<kind>/<sub>/*.ndjson` (from the step above).
2. **A few download-tool gap re-exports** - large bulk dumps pulled specifically
   to close holes the first export missed (e.g. a ~260-day `nycrail` posts hole,
   plus `fuckcars`, `EastVillage`, and `MicromobilityNYC`). Together these added
   ~27k records.

Both use the identical merge: read existing Parquet -> concat new records
(**existing first, so they win on dedup**) -> drop duplicate `id`s -> re-run
`prepare_for_parquet` -> atomic write. Across the two patch rounds this added
~167k then ~47k net-new records; the tool-gap files added ~27k more.

In [12]:
def merge_records_into_parquet(sub, kind, new_df):
    """Fold new rows into parquet/<kind>/<sub>.parquet. Existing rows win on dedup."""
    cols = COLS[kind]
    new_df = new_df[[c for c in cols if c in new_df.columns]]
    path = PARQUET / kind / f"{sub}.parquet"
    existing = pd.read_parquet(path) if path.exists() else pd.DataFrame()
    n_exist = len(existing)
    combined = pd.concat([existing, new_df], ignore_index=True) if n_exist else new_df
    before = len(combined)
    combined.drop_duplicates(subset=["id"], keep="first", inplace=True)
    after = len(combined)
    combined = prepare_for_parquet(combined)
    write_parquet_safe(combined, path)
    return n_exist, after - n_exist, before - after   # (existing, net_new, dupes)


def merge_patch_data():
    added = 0
    for kind in ("posts", "comments"):
        kdir = PATCH_DIR / kind
        if not kdir.exists():
            continue
        for sub_dir in sorted(p for p in kdir.iterdir() if p.is_dir()):
            files = sorted(sub_dir.glob("*.ndjson"))
            dfs = [d for d in (read_jsonl(f, COLS[kind]) for f in files) if not d.empty]
            if not dfs:
                continue
            new = pd.concat(dfs, ignore_index=True)
            _, net, dup = merge_records_into_parquet(sub_dir.name, kind, new)
            added += net
            print(f"  {kind}/{sub_dir.name}: +{net:,} net new ({dup:,} dupes dropped)")
    print(f"Patch merge: +{added:,} net new.\n")


# Download-tool gap re-exports: (subreddit, kind, file)
MERGES = [
    ("EastVillage",      "comments", TOOL_DIR / "r_EastVillage_comments-gap.jsonl"),
    ("fuckcars",         "comments", TOOL_DIR / "r_fuckcars_comments-gap.jsonl"),
    ("fuckcars",         "posts",    TOOL_DIR / "r_fuckcars_posts-gap.jsonl"),
    ("nycrail",          "posts",    TOOL_DIR / "r_nycrail_posts-gap.jsonl"),
    ("MicromobilityNYC", "comments", TOOL_DIR / "r_r_MicromobilityNYC_comments.jsonl"),
    ("MicromobilityNYC", "posts",    TOOL_DIR / "r_r_MicromobilityNYC_posts.jsonl"),
]


def merge_tool_gaps():
    added = 0
    for sub, kind, f in MERGES:
        if not f.exists() or f.stat().st_size == 0:
            print(f"  (skip {f.name}: missing/empty)")
            continue
        new = read_jsonl(f, COLS[kind])
        if new.empty:
            continue
        _, net, dup = merge_records_into_parquet(sub, kind, new)
        added += net
        print(f"  {kind}/{sub}: +{net:,} net new ({dup:,} dupes dropped)")
    print(f"Tool-gap merge: +{added:,} net new.\n")


if RUN_MERGE:
    merge_patch_data()
    merge_tool_gaps()
else:
    print("RUN_MERGE is False -- skipping merges.")

RUN_MERGE is False -- skipping merges.


## 12 - Step 7: Combine into the two final mega-files

Last step: concatenate every per-subreddit Parquet into
`parquet/all_posts.parquet` and `parquet/all_comments.parquet`.

The per-subreddit files do not share an identical schema - they were built from
different sources and pandas/pyarrow versions, so we see `string` vs
`large_string`, different column orders, the odd always-null column stored as
`double`, etc. We fix that with one **canonical PyArrow schema** per kind and a
`cast_to_schema()` that reorders columns, casts types (`safe=False`), fills
missing columns with nulls, and back-fills the `subreddit` column from the
filename. All text is `large_string` so the 2 GB comments file cannot overflow
32-bit string offsets. We stream in 500k-row batches through a single
`ParquetWriter`, keeping memory flat.

In [13]:
CHUNK_ROWS = 500_000
LS, I64, F64, BOOL = pa.large_string(), pa.int64(), pa.float64(), pa.bool_()

POST_SCHEMA = pa.schema([
    pa.field("id", LS), pa.field("name", LS), pa.field("created_utc", I64),
    pa.field("retrieved_on", I64), pa.field("title", LS), pa.field("selftext", LS),
    pa.field("previous_selftext", LS), pa.field("url", LS), pa.field("domain", LS),
    pa.field("permalink", LS), pa.field("author", LS), pa.field("author_fullname", LS),
    pa.field("author_created_utc", F64), pa.field("subreddit", LS),
    pa.field("subreddit_id", LS), pa.field("subreddit_subscribers", I64),
    pa.field("score", I64), pa.field("ups", I64), pa.field("upvote_ratio", F64),
    pa.field("num_comments", I64), pa.field("is_self", BOOL), pa.field("_media_type", LS),
    pa.field("post_hint", LS), pa.field("over_18", BOOL), pa.field("locked", BOOL),
    pa.field("stickied", BOOL), pa.field("distinguished", LS), pa.field("edited", I64),
    pa.field("removed_by_category", LS), pa.field("link_flair_text", LS),
    pa.field("poll_data", LS),
])

COMMENT_SCHEMA = pa.schema([
    pa.field("id", LS), pa.field("name", LS), pa.field("created_utc", I64),
    pa.field("retrieved_on", I64), pa.field("body", LS), pa.field("permalink", LS),
    pa.field("author", LS), pa.field("author_fullname", LS),
    pa.field("author_created_utc", F64), pa.field("subreddit", LS),
    pa.field("subreddit_id", LS), pa.field("link_id", LS), pa.field("parent_id", LS),
    pa.field("score", I64), pa.field("ups", I64), pa.field("controversiality", I64),
    pa.field("edited", I64), pa.field("distinguished", LS), pa.field("is_submitter", BOOL),
    pa.field("stickied", BOOL), pa.field("locked", BOOL), pa.field("collapsed", BOOL),
    pa.field("removed_by_category", LS),
])

SCHEMAS = {"posts": POST_SCHEMA, "comments": COMMENT_SCHEMA}


def cast_to_schema(table, target, sub):
    """Reorder + cast a table to the target schema; fill missing/odd columns safely."""
    arrays = []
    for field in target:
        if field.name == "subreddit":
            col = table.column("subreddit") if "subreddit" in table.schema.names else None
            if col is None or col.null_count == len(col):
                arrays.append(pa.array([sub] * len(table), type=LS))
            elif col.null_count > 0:
                filled = [v.as_py() if v.as_py() is not None else sub for v in col]
                arrays.append(pa.array(filled, type=LS))
            else:
                arrays.append(col.cast(LS, safe=False))
        elif field.name not in table.schema.names:
            arrays.append(pa.nulls(len(table), type=field.type))
        else:
            col = table.column(field.name)
            if col.type == field.type:
                arrays.append(col)
            else:
                try:
                    arrays.append(col.cast(field.type, safe=False))
                except Exception:
                    try:
                        arrays.append(col.cast(pa.string(), safe=False).cast(field.type, safe=False))
                    except Exception:
                        arrays.append(pa.nulls(len(table), type=field.type))
    return pa.table({f.name: arr for f, arr in zip(target, arrays)}, schema=target)


def combine(kind):
    in_dir = PARQUET / kind
    out_path = PARQUET / f"all_{kind}.parquet"
    tmp_path = PARQUET / f"all_{kind}.parquet.tmp"
    target = SCHEMAS[kind]
    files = sorted(in_dir.glob("*.parquet"))
    if not files:
        print(f"  no files in {in_dir}")
        return
    print(f"\n===== {kind.upper()} ({len(files)} subreddits) =====")
    writer, total = pq.ParquetWriter(tmp_path, target, compression="zstd"), 0
    for pf in files:
        sub = pf.stem
        reader = pq.ParquetFile(pf)
        for batch in reader.iter_batches(batch_size=CHUNK_ROWS):
            tbl = cast_to_schema(pa.Table.from_batches([batch]), target, sub)
            writer.write_table(tbl)
            total += len(tbl)
        print(f"  [{sub}] {reader.metadata.num_rows:,} rows  ok")
    writer.close()
    os.replace(tmp_path, out_path)
    size_mb = out_path.stat().st_size / 1_048_576
    print(f"  -> all_{kind}.parquet  ({total:,} rows, {size_mb:.1f} MB)")


if RUN_COMBINE:
    for kind in ("posts", "comments"):
        combine(kind)
    print("\nCombine done.")
else:
    print("RUN_COMBINE is False -- skipping the final combine.")

RUN_COMBINE is False -- skipping the final combine.


## 13 - Final verification

Load the two mega-files and confirm the totals and the per-subreddit breakdown.
(Reads only - runs whenever the files exist.)

In [14]:
for kind in ("posts", "comments"):
    path = PARQUET / f"all_{kind}.parquet"
    if not path.exists():
        print(f"(no {path.name} yet)")
        continue
    pf = pq.ParquetFile(path)
    print(f"all_{kind}.parquet: {pf.metadata.num_rows:,} rows, "
          f"{path.stat().st_size / 1_048_576:.1f} MB, {pf.metadata.num_columns} cols")

p = PARQUET / "all_comments.parquet"
if p.exists():
    subs = pq.read_table(p, columns=["subreddit"]).column("subreddit").to_pandas()
    vc = subs.value_counts()
    print(f"\n{len(vc)} subreddits in all_comments.parquet. Top 15 by volume:\n")
    print(vc.head(15).to_string())

all_posts.parquet: 996,115 rows, 213.6 MB, 31 cols
all_comments.parquet: 18,408,220 rows, 1992.9 MB, 23 cols

58 subreddits in all_comments.parquet. Top 15 by volume:

subreddit
nyc            2177243
fuckcars       1962585
newjersey      1897458
AskNYC         1895712
Connecticut    1848310
longisland     1188778
jerseycity      764790
nycrail         595497
Brooklyn        571352
transit         537968
astoria         531868
newyorkcity     475159
Westchester     367616
NYCbike         282690
Hoboken         264431


## 14 - Summary & known, unfixable gaps

**Result** (after the full pipeline):

| File | Rows | Size |
|------|-----:|-----:|
| `all_posts.parquet`    |    996,115 |   213.6 MB |
| `all_comments.parquet` | 18,408,220 | 1,992.9 MB |

55 subreddits in posts, 58 in comments, every row carries `subreddit`, dedup on
`id` throughout, window 2023-01-01 -> 2026-04-30.

**Gaps that remain by nature, not oversight:**

- **June 2023 Reddit blackout** (Jun 12-18) - subs were private; no data exists.
- **Dormant stretches** - e.g. EastVillage Dec 2023 -> May 2024; the sub was quiet.
- **Sparse small subs** - naturally intermittent days.
- A few extremely dense comment-days that repeatedly timed out (HTTP 422) are
  filled to within a small tolerance.

**Reproducing from scratch:** place the download-tool JSONL in
`searchtool download/`, then set the `RUN_*` flags to `True` and run top to
bottom. The one thing this notebook deliberately *summarizes* rather than replays
verbatim is the iterative re-downloading of a few oversized tool files (the
`...2` / `...3` / `...-gap` siblings); the consolidated `PATCHES` list and `MERGES`
mapping above reach the same final coverage in a single pass.